In [1]:
import numpy as np
import pandas as pd

In [2]:
# Load & Clean
df = pd.read_csv("amazon_products.csv")

In [3]:
df.head()

,asin,title,imgUrl,productURL,stars,reviews,price,listPrice,category_id,isBestSeller,boughtInLastMonth
0,B014TMV5YE,"Sion Softside Expandable Roller Luggage, Black...",https://m.media-amazon.com/images/I/815dLQKYIY...,https://www.amazon.com/dp/B014TMV5YE,4.5,0,139.99,0.00,104,False,2000
1,B07GDLCQXV,Luggage Sets Expandable PC+ABS Durable Suitcas...,https://m.media-amazon.com/images/I/81bQlm7vf6...,https://www.amazon.com/dp/B07GDLCQXV,4.5,0,169.99,209.99,104,False,1000
2,B07XSCCZYG,Platinum Elite Softside Expandable Checked Lug...,https://m.media-amazon.com/images/I/71EA35zvJB...,https://www.amazon.com/dp/B07XSCCZYG,4.6,0,365.49,429.99,104,False,300
3,B08MVFKGJM,Freeform Hardside Expandable with Double Spinn...,https://m.media-amazon.com/images/I/91k6NYLQyI...,https://www.amazon.com/dp/B08MVFKGJM,4.6,0,291.59,354.37,104,False,400
4,B01DJLKZBA,Winfield 2 Hardside Expandable Luggage with Sp...,https://m.media-amazon.com/images/I/61NJoaZcP9...,https://www.amazon.com/dp/B01DJLKZBA,4.5,0,174.99,309.99,104,False,400


In [4]:
df.tail()

,asin,title,imgUrl,productURL,stars,reviews,price,listPrice,category_id,isBestSeller,boughtInLastMonth
1426332,B00R3LIKCO,American Flag Patriotic USA Classic 5 Panel Me...,https://m.media-amazon.com/images/I/71PDJFz6AA...,https://www.amazon.com/dp/B00R3LIKCO,4.2,0,14.95,0.00,112,False,0
1426333,B098BQ7ZQ3,Men's Baseball Cap - H2O-DRI Line Up Curved Br...,https://m.media-amazon.com/images/I/812Tycexs4...,https://www.amazon.com/dp/B098BQ7ZQ3,4.4,0,33.99,0.00,112,False,0
1426334,B07X1MVNT1,[4 Pack] Adjustable Eyeglasses and Sunglasses ...,https://m.media-amazon.com/images/I/61vvYW1S9J...,https://www.amazon.com/dp/B07X1MVNT1,3.6,0,8.54,0.00,112,False,0
1426335,B08XLBG8V9,Ax2002 Aviator Sunglasses,https://m.media-amazon.com/images/I/51+yjD4F1x...,https://www.amazon.com/dp/B08XLBG8V9,4.5,0,54.36,57.39,112,False,0
1426336,B07GH67QC8,in Hoc Signo Vinces Knights Templar Masonic Em...,https://m.media-amazon.com/images/I/91Kt2KQf0E...,https://www.amazon.com/dp/B07GH67QC8,4.9,0,18.79,0.00,112,False,0


In [5]:
df.shape

(1426337, 11)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1426337 entries, 0 to 1426336
Data columns (total 11 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   asin               1426337 non-null  object 
 1   title              1426336 non-null  object 
 2   imgUrl             1426337 non-null  object 
 3   productURL         1426337 non-null  object 
 4   stars              1426337 non-null  float64
 5   reviews            1426337 non-null  int64  
 6   price              1426337 non-null  float64
 7   listPrice          1426337 non-null  float64
 8   category_id        1426337 non-null  int64  
 9   isBestSeller       1426337 non-null  bool   
 10  boughtInLastMonth  1426337 non-null  int64  
dtypes: bool(1), float64(3), int64(3), object(4)
memory usage: 110.2+ MB


In [7]:
# It removes any rows where the product name is missing (useless for recommendations).
# and also ensures no product appears twice.
df = df.dropna(subset = ["title"]).drop_duplicates(subset = ["asin"])

In [8]:
df.shape

(1426336, 11)

In [9]:
df.isna().sum()

asin                 0
title                0
imgUrl               0
productURL           0
stars                0
reviews              0
price                0
listPrice            0
category_id          0
isBestSeller         0
boughtInLastMonth    0
dtype: int64

In [10]:
df.describe()

,stars,reviews,price,listPrice,category_id,boughtInLastMonth
count,1.426336e+06,1.426336e+06,1.426336e+06,1.426336e+06,1.426336e+06,1.426336e+06
mean,3.999511e+00,1.807509e+02,4.337541e+01,1.244917e+01,1.237410e+02,1.419824e+02
std,1.344293e+00,1.761454e+03,1.302893e+02,4.611200e+01,7.311271e+01,8.362722e+02
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00
25%,4.100000e+00,0.000000e+00,1.199000e+01,0.000000e+00,6.500000e+01,0.000000e+00
50%,4.400000e+00,0.000000e+00,1.995000e+01,0.000000e+00,1.200000e+02,0.000000e+00
75%,4.600000e+00,0.000000e+00,3.599000e+01,0.000000e+00,1.760000e+02,5.000000e+01
max,5.000000e+00,3.465630e+05,1.973181e+04,9.999900e+02,2.700000e+02,1.000000e+05


In [11]:
# Then numeric columns are safely converted using pd.to_numeric(..., errors="coerce") — this handles cases where a value might be a string like "N/A" instead of crashing
# Missing stars and price get filled with the median (more robust than mean against outliers).
for col in ["stars", "price"]:
    df[col] = pd.to_numeric(df[col], errors = "coerce").fillna(df[col].median())

for col in ["reviews", "boughtInLastMonth"]:
    df[col] = pd.to_numeric(df[col], errors = "coerce").fillna(0)

In [12]:
# In this, sample(min(50000, len(df))) picks up to 50k rows randomly — the min() guard prevents a crash if your dataset is smaller than 50k.
df.sample(min(50000, len(df)), random_state=42).reset_index(drop = True)

,asin,title,imgUrl,productURL,stars,reviews,price,listPrice,category_id,isBestSeller,boughtInLastMonth
0,B0BKZNNL1V,"LOVEVOOK Diaper Bag Backpack, Diaper Backpack ...",https://m.media-amazon.com/images/I/718pt0oLES...,https://www.amazon.com/dp/B0BKZNNL1V,4.6,0,36.99,0.00,36,False,0
1,B0968QPTK8,Seloom 148 PCS Reflective Mailbox Numbers and ...,https://m.media-amazon.com/images/I/71Sr2+igJf...,https://www.amazon.com/dp/B0968QPTK8,4.5,286,14.89,19.99,211,False,50
2,B00NIAULVC,"Spry Xylitol Toothpaste 5oz, Fluoride Toothpas...",https://m.media-amazon.com/images/I/51I687iehe...,https://www.amazon.com/dp/B00NIAULVC,4.7,0,8.49,8.99,126,False,900
3,B0794ZFTDB,2 Pieces Rhinestone Bow Ties Banquet Bowties M...,https://m.media-amazon.com/images/I/81G1Dc1uZu...,https://www.amazon.com/dp/B0794ZFTDB,4.4,0,12.99,13.99,112,False,50
4,B075PTHVKN,Mielle Organics Pomegranate & Honey Leave-In C...,https://m.media-amazon.com/images/I/71RNqb6TfJ...,https://www.amazon.com/dp/B075PTHVKN,4.7,0,12.52,14.99,47,False,10000
...,...,...,...,...,...,...,...,...,...,...,...
49995,B0C5LJ3PS9,"Car Ceiling Cargo Net Pocket, Upgraded (NO SAG...",https://m.media-amazon.com/images/I/91u9iJ2gjh...,https://www.amazon.com/dp/B0C5LJ3PS9,4.1,31,16.99,17.99,24,False,100
49996,B0001XVUFA,Men's MQ24-9B Classic Analog Watch,https://m.media-amazon.com/images/I/81P4k199br...,https://www.amazon.com/dp/B0001XVUFA,4.4,0,15.95,22.95,113,False,700
49997,B0CC87MNYQ,Unisex Kids Pajamas Set Tee and Pant 2-Piece P...,https://m.media-amazon.com/images/I/91ocDh71c3...,https://www.amazon.com/dp/B0CC87MNYQ,0.0,0,23.99,0.00,91,False,0
49998,B099JK7ZT3,"12 Colors Nail Art Glitter Sequins, Irregular ...",https://m.media-amazon.com/images/I/71ViRFvPas...,https://www.amazon.com/dp/B099JK7ZT3,4.5,414,7.59,0.00,51,False,50


The four columns stars, reviews, price, and boughtInLastMonth are on completely different scales — reviews could be in the thousands while stars is just 1–5. If you fed raw numbers to KNN or KMeans, reviews would dominate simply because its numbers are bigger. MinMaxScaler squishes every column to the range [0, 1], so all four contribute equally.

In [13]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans

In [14]:
# Feature Scaling
feats = ['stars', 'reviews', 'price', 'boughtInLastMonth']
scaled = MinMaxScaler().fit_transform(df[feats])
scaled_df = pd.DataFrame(scaled, columns = feats)

In [ ]:
# Models
knn = NearestNeighbors(n_neighbors = 6, metric = 'euclidean').fit(scaled)
df['cluster'] = KMeans(n_clusters = 10, random_state = 42, n_init = 10).fit_predict(scaled)
df['pop_score'] = scaled_df['stars']*0.4 + scaled_df['reviews']*0.35 + scaled_df['boughtInLastMonth']*0.25

In [ ]:
import pickle

# SAVE FILES

with open("Recommendation_System.pkl", "wb") as f:
    pickle.dump({
        "df":     df,
        "scaled": scaled,
        "knn":    knn,
    }, f)
 
print("✅ Recommendation_System.pkl saved successfully!")